In [ ]:
'''!pip install -q mlxtend'''

In [7]:
import os
import pandas as pd
from google.cloud import bigquery
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore')

In [8]:
%reload_ext watermark
%watermark -a "Matheus dos Anjos" --iversions

Author: Matheus dos Anjos

mlxtend: 0.24.0
pandas : 2.3.3



In [9]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "cap04-486320-a6023474858c.json"

In [10]:
client = bigquery.Client()

In [13]:
query = """ 
SELECT
      f.Cliente_ID,
      f.Produto_ID,
      p.Nome AS Produto,
      f.Quantidade
FROM `cap04-486320.Projeto2.fato_venda` f
JOIN `cap04-486320.Projeto2.dim_produto` p
ON f.Produto_ID = p.Produto_ID
"""

In [14]:
df = client.query(query).to_dataframe()

In [15]:
df.head(3)

,Cliente_ID,Produto_ID,Produto,Quantidade
0,8,2,Produto B,14
1,10,3,Produto C,3
2,5,4,Produto D,7


In [16]:
df.shape

(50, 4)

In [19]:
basket = df.pivot_table(index = 'Cliente_ID',
                        columns = 'Produto',
                        values = 'Quantidade',
                        aggfunc = 'sum').fillna(0)

In [20]:
basket

Produto,Produto A,Produto B,Produto C,Produto D,Produto E,Produto F,Produto G,Produto H,Produto I,Produto J
Cliente_ID,,,,,,,,,,
1,4,16,0,35,0,0,3,17,0,3
2,0,0,17,0,1,17,0,0,17,2
3,0,15,0,0,17,0,0,0,5,0
4,0,0,8,0,6,0,0,0,18,0
5,0,0,6,7,14,5,0,0,0,18
6,12,2,12,0,10,0,1,1,8,0
7,0,0,0,0,1,2,18,0,0,14
8,0,24,0,0,19,0,19,0,0,16
9,0,0,0,0,0,0,16,14,0,0


In [21]:
basket = basket.applymap(lambda x:1 if x > 0 else 0)

In [22]:
basket

Produto,Produto A,Produto B,Produto C,Produto D,Produto E,Produto F,Produto G,Produto H,Produto I,Produto J
Cliente_ID,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,1
2,0,0,1,0,1,1,0,0,1,1
3,0,1,0,0,1,0,0,0,1,0
4,0,0,1,0,1,0,0,0,1,0
5,0,0,1,1,1,1,0,0,0,1
6,1,1,1,0,1,0,1,1,1,0
7,0,0,0,0,1,1,1,0,0,1
8,0,1,0,0,1,0,1,0,0,1
9,0,0,0,0,0,0,1,1,0,0


In [23]:
frequent_itemsets = apriori(basket, min_support=0.3, use_colnames=True)

In [24]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

In [26]:
rules.head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Produto B),(Produto E),0.4,0.7,0.3,0.75,1.071429,1.0,0.02,1.20,0.111111,0.375000,0.166667,0.589286
1,(Produto B),(Produto G),0.4,0.6,0.3,0.75,1.250000,1.0,0.06,1.60,0.333333,0.428571,0.375000,0.625000
2,(Produto C),(Produto E),0.5,0.7,0.4,0.80,1.142857,1.0,0.05,1.50,0.250000,0.500000,0.333333,0.685714
3,(Produto I),(Produto C),0.4,0.5,0.3,0.75,1.500000,1.0,0.10,2.00,0.555556,0.500000,0.500000,0.675000
4,(Produto C),(Produto I),0.5,0.4,0.3,0.60,1.500000,1.0,0.10,1.50,0.666667,0.500000,0.333333,0.675000
5,(Produto F),(Produto E),0.3,0.7,0.3,1.00,1.428571,1.0,0.09,inf,0.428571,0.428571,1.000000,0.714286
6,(Produto I),(Produto E),0.4,0.7,0.4,1.00,1.428571,1.0,0.12,inf,0.500000,0.571429,1.000000,0.785714
7,(Produto J),(Produto E),0.5,0.7,0.4,0.80,1.142857,1.0,0.05,1.50,0.250000,0.500000,0.333333,0.685714
8,(Produto F),(Produto J),0.3,0.5,0.3,1.00,2.000000,1.0,0.15,inf,0.714286,0.600000,1.000000,0.800000
9,(Produto J),(Produto F),0.5,0.3,0.3,0.60,2.000000,1.0,0.15,1.75,1.000000,0.600000,0.428571,0.800000


Aqui está uma interpretação detalhada das métricas para as regras de associação listadas:

**Antecedents (Antecedentes)**: Os itens ou conjunto de itens no lado esquerdo da regra. Eles representam a condição inicial da associação.

**Consequents (Consequentes)**: Os itens ou conjunto de itens no lado direito da regra. Eles são o resultado esperado quando os antecedentes ocorrem.

**Antecedent Support (Suporte do Antecedente)**: A frequência relativa com que o antecedente ocorre no conjunto de dados.

**Consequent Support (Suporte do Consequente)**: A frequência relativa com que o consequente ocorre no conjunto de dados.

**Support (Suporte)**: A frequência com que ambos, o antecedente e o consequente, ocorrem juntos nas transações. 

**Confidence (Confiança)**: Mede a probabilidade de o consequente ocorrer quando o antecedente ocorre. É calculado como a razão entre o suporte da regra e o suporte do antecedente. 

**Lift (Efeito ou Elevação)**: Avalia a força da associação entre antecedente e consequente comparando com o que seria esperado se eles fossem independentes. 

**Leverage**: Mede o quão mais frequentemente o antecedente e o consequente ocorrem juntos do que o esperado se fossem independentes. Um valor positivo indica uma associação útil.

**Conviction**: Mede a independência das ocorrências de antecedentes e consequentes, comparando a frequência real de ocorrência do antecedente sem o consequente com a frequência esperada. Um valor alto indica uma relação de dependência forte. Valores de "inf" indicam que a consequente praticamente nunca ocorre sem o antecedente.

**Zhang's Metric**: Avalia a força da associação em termos de independência, sendo mais robusta contra o viés do suporte. Um valor próximo de 1 indica uma forte relação positiva, enquanto valores mais baixos indicam uma associação fraca.

# Lembrando que são porcentagens